In [2]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)



import pandas as pd
import numpy as np
import re
import random 
import matplotlib.pyplot as plt

from wordcloud import WordCloud
import spacy

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# Load the small English pipeline
nlp = spacy.load("en_core_web_sm")


ModuleNotFoundError: No module named 'wordcloud'

In [ ]:
! python -m spacy download en_core_web_sm

In [ ]:
raw_documents = [
    # Technology
    "The new smartphone model has an amazing high-definition camera.",
    "Data privacy is a huge concern in modern social media apps.",
    "Artificial intelligence is changing the way software is developed.",
    "Quantum computing might revolutionize the entire technological landscape.",
    "The system is connecting to the server while other devices connect automatically.",
    "Stable connections are vital, but his connection remained very weak.",
    "The programmer is programming a new application for mobile users.",
    "Recommendation engines use your browsing history to suggest products and content.",
    "Supercomputers compute complex data faster than a regular computer can.",
    "There are certainly elements like artificial intelligence that have expanded how we use our smartphones.",
    "Software engineers apply engineering principles and knowledge of programming languages to build software solutions for end users.",
    "Artificial intelligence is the simulation of human intelligence in machines that are programmed to think like humans and mimic their actions.",
    "Strong encryption is the backbone of modern cybersecurity protocols.",
    "GPS navigation helps users navigate even when they are navigating difficult terrain.",
    "The navigator signaled that the navigation system was finally navigated correctly.",
    "5G networks promise faster download speeds and lower latency for mobile users.",
    "The smartphone combined technologies that once required multiple separate devices.",
    "Battery optimization technology extends usage time between charges.",
    "Algorithms determine what content users see, creating personalized but filtered realities.",
    "My phone automatically organized all my photos by face and location.",
 
    # Cooking
    "Olive oil is an all-purpose oil used in regular daily cooking",
    "The secret ingredient in my grandmother's cake recipe is nutmeg.",
    "Stir-frying is one of the quickest and easiest cooking methods, and it works great with vegetables, proteins, and even starches like potatoes.",
    "Baking ingredients are used for baking, although they can also be used for cooking sometimes."
    "Prepare the ingredients like onions and garlic before starting the stir-fry.",
    "The smoked variety imparts a deep, smoky flavor to roasted vegetables, meats, and stews.",
    "The chef fried the fish until it was perfectly fried and crispy.",
    "When you are baking, remember that baked goods continue baking after leaving the oven.",
    "She seasoned the meat after seasoning the vegetables with salt.",
    "The recipe calls for three different seasonings and spices.",
    "Maybe you like your garlic flavor a little stronger, in which case you can add two or three cloves, or perhaps you might want to use olive oil instead of butter when you sauté something to give the dish a lighter, healthier tone.",
    "A small amount of oil is used in a hot pan at medium-high heat to cook the food, resulting in a golden, usually crisp exterior that locks in moisture.",
    "If you are whisksing by hand, your arm might get tired quite quickly.",
    "The restaurant specializes in poached eggs served with a side of hollandaise.",
    "Plain flour is essential in baking, thickening sauces, and giving food crusts",
    "They had poached several salmon fillets before the dinner rush started.",
    "Seasoning and a light coating of oil help boost crispness and enhance taste.",
    "Marinating beforehand enhances tenderness and taste, whereas grill marks give food a distinctive, appetizing look.",
    "For vegetables, avoid overboiling to preserve color and nutrients, and try using a lid to bring water to a boil faster."
 
    # Politics
    "The prime minister addressed the parliament regarding the new trade agreement.",
    "Voters must consider all aspects of the referendum before casting a ballot.",
    "The government announced a significant decrease in environmental taxes.",
    "Local councils are meeting to discuss urban development plans.",
    "Citizens are voting in the local mayoral election today.",
    "High energy prices have been a consistent theme in discussions among leaders for months",
    "The government published a white paper policy document this Monday.",
    "Good governance is required for any governor to be successful.",
    "The committee elected a leader after the previous elections were voided.",
    "An elective body manages how the electoral process is handled.",
    "The legislative body is debating the merits of the proposed healthcare bill.",
    "A peaceful protest can be a powerful tool for driving social change.",
    "The protestor was arrested while protesting alongside other determined protestors.",
    "A bill is a draft of a new law or a change to an existing law, presented to Parliament.",
    "Following the Cabinet's approval, the Department of Justice drafts a bill.",
    "To participate in political elections, you must be registered to vote.",
    "Political analysts suggest that the recent policy shift could influence the upcoming election results.",
    "Public opinion polls indicate a significant generational divide on climate change policies.",
    "The opposition party criticized the administration's handling of the economic crisis.",
    "The rise of populist movements across Europe reflects growing public dissatisfaction with traditional parties.",
]
 
print(f"Loaded a corpus of {len(raw_documents)}documents.")
print("Exemple raw text:\",raw_documets[8]")

Loaded a corpus of 57documents.
Exemple raw text:",raw_documets[8]


In [ ]:
# Scenario 1: Raw (Minimal cleaning)
def preprocess_raw(text):
    return text.lower()
 
# Scenario 2: No Lemma (Stopwords removed only)
def preprocess_no_lemma(text):
    doc = nlp(text)
    return " ".join([t.lower_ for t in doc if not t.is_stop and t.is_alpha])
 
# Scenario 3: Full spaCy (Lemmatized + Stopwords removed)
# We use a list comprehension to:
    # - Lowercase (token.lemma_.lower())
    # - Remove Stop words (token.is_stop)
    # - Remove Punctuation (token.is_alpha)
    # - Extract the Lemma (token.lemma_)
def preprocess_cleaned(text):
    doc = nlp(text)
    return " ".join([t.lemma_.lower() for t in doc if not t.is_stop and t.is_alpha])
 
# Scenario 4: Keep only the NOUNS
def preprocess_nouns_only(text):
    doc = nlp(text)
    # Only keep Nouns - these are usually the "Topics"
    return " ".join([t.lemma_.lower() for t in doc if t.pos_ == "NOUN" and not t.is_stop])

In [ ]:
data_raw = [preprocess_raw(d) for d in raw_documents] # This is scenario 1
data_no_lemma = [preprocess_no_lemma(d) for d in raw_documents] # This is scenario 2
data_cleaned = [preprocess_cleaned(d) for d in raw_documents] # This is scenario 3
data_nouns = [preprocess_nouns_only(d) for d in raw_documents] # This is scenario 4

index_to_show = 9
print(f"--- Comparison of Pre-processing (Doc {index_to_show}) ---")
print(f"Raw:       {data_raw[index_to_show]}")
print(f"No_Lemma:  {data_no_lemma[index_to_show]}")
print(f"Cleaned:   {data_cleaned[index_to_show]}")
print(f"Nouns:      {data_nouns[index_to_show]}")

NameError: name 'nlp' is not defined

In [ ]:
def run_lda_and_plot(data, title, save_model=False, model_name=None):
    # 1. Vectorize (Convert text to numbers)
    vectorizer = CountVectorizer(max_features=1000)
    tf = vectorizer.fit_transform(data)
    words = vectorizer.get_feature_names_out()
 
    # 2. Fit LDA Model (Finding 3 topics)
    lda = LatentDirichletAllocation(n_components=3, max_iter=10, random_state=42)
    lda.fit(tf)
 
    # 3. Plotting Word Clouds
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(title, fontsize=16, fontweight='bold')
 
    for topic_idx, topic in enumerate(lda.components_):
        # Create a dictionary of word frequencies for the cloud
        word_freq = {words[i]: topic[i] for i in topic.argsort()[:-20 - 1:-1]}
        
        wc = WordCloud(background_color='white', colormap='tab10').generate_from_frequencies(word_freq)
        
        axes[topic_idx].imshow(wc, interpolation='bilinear')
        axes[topic_idx].set_title(f"Topic {topic_idx}")
        axes[topic_idx].axis('off')
    
    plt.tight_layout()
    plt.subplots_adjust(top=0.85)
    plt.show()
    
    # Save model
    if save_model and model_name:
        with open(f'{model_name}_lda.pkl', 'wb') as f:
            pickle.dump(lda, f)
        with open(f'{model_name}_vectorizer.pkl', 'wb') as f:
            pickle.dump(vectorizer, f)
        print(f"Models saved as {model_name}_lda.pkl and {model_name}_vectorizer.pkl")
    
    return lda, vectorizer
 
# Train and save all models (run this once)
lda_raw, vectorizer_raw = run_lda_and_plot(data_raw, "SCENARIO 1: RAW TEXT",
                                           save_model=True, model_name="raw")
lda_no_lemma, vectorizer_no_lemma = run_lda_and_plot(data_no_lemma, "SCENARIO 2: STOPWORDS REMOVED (NO LEMMA)",
                                                     save_model=True, model_name="no_lemma")
lda_cleaned, vectorizer_cleaned = run_lda_and_plot(data_cleaned, "SCENARIO 3: FULL CLEANING",
                                                   save_model=True, model_name="cleaned")
lda_nouns, vectorizer_nouns = run_lda_and_plot(data_nouns, "SCENARIO 4: ONLY NOUNS",
                                               save_model=True, model_name="nouns")
 



NameError: name 'CountVectorizer' is not defined